# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

- [FAIR^2 Croissant Schema JSON-LD file](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import matplotlib.pyplot as plt

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}\n\nIdentifier: {metadata.identifier}\n\nPublished: {metadata.datePublished}\n\nLicense: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

In [ ]:
# List record sets and their IDs
print("Available Record Sets:")
for rs in metadata.recordSet:
    print(f"- {rs['@id']}: {rs.get('name', rs['@id'])}")

# Show fields for each record set
for rs in metadata.recordSet:
    print(f"\nRecord Set: {rs['@id']}")
    if 'field' in rs:
        print("Fields:")
        for field in rs['field']:
            print(f"  - {field['@id']}: {field.get('name', field['@id'])} (type: {field.get('dataType', 'unknown')})")
    else:
        print("  No fields found.")

In [ ]:
# Display a sample of records from each record set using their @id
for rs in metadata.recordSet:
    rs_id = rs['@id']
    print(f"\nSample Records from Record Set: {rs_id}")
    try:
        for idx, record in enumerate(dataset.records(record_set=rs_id)):
            print(record)
            if idx == 2:  # print first 3 records
                break
    except Exception as e:
        print(f"  Could not load records for {rs_id}: {e}")

## 3. Data Extraction
Load data from all available record sets into DataFrames for analysis.
Use the record set and field `@id`s from the previous overview.

In [ ]:
# Extract data from each record set using their @id
record_sets_ids = [rs['@id'] for rs in metadata.recordSet]
dataframes = {}

for record_set_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nRecord Set: {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head())
    except Exception as e:
        print(f"  Could not load records for {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll select the main tabular record set containing clinical and pathological variables. Use the field `@id` for analysis.

In [ ]:
# Select the main tabular record set
# If multiple record sets exist, pick the one with the largest DataFrame
main_rs_id = None
max_rows = 0
for rs_id, df in dataframes.items():
    if df.shape[0] > max_rows:
        main_rs_id = rs_id
        max_rows = df.shape[0]

main_df = dataframes[main_rs_id]
print(f"Main Record Set (@id): {main_rs_id}, {main_df.shape[0]} rows")

# Select numeric fields for analysis by type
fields_info = {}
for rs in metadata.recordSet:
    if rs['@id'] == main_rs_id:
        if 'field' in rs:
            for field in rs['field']:
                fields_info[field['@id']] = field.get('dataType', 'unknown')
numeric_fields = [fid for fid, dtype in fields_info.items() if dtype in ['schema:Integer', 'schema:Float', 'schema:Number']]
print(f"Numeric fields: {numeric_fields}")

# Use first numeric field (if any) for demonstration
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    numeric_field = numeric_field_id
    # Ensure the field exists in DataFrame
    if numeric_field in main_df.columns:
        # Remove missing values
        filtered_df = main_df[main_df[numeric_field].notnull()]
        threshold = filtered_df[numeric_field].median()
        filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a categorical field (pick the first string field)
        string_fields = [fid for fid, dtype in fields_info.items() if dtype == 'schema:Text']
        group_field = None
        for fid in string_fields:
            if fid in filtered_df.columns:
                group_field = fid
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            print(grouped_df.head())
        else:
            print("No grouping field found.")
    else:
        print(f"Numeric field {numeric_field} not found in main dataframe.")
else:
    print("No numeric fields found in main record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualization, if there is a numeric field and grouping field
if 'filtered_df' in locals() and not filtered_df.empty and group_field and numeric_field:
    plt.figure(figsize=(8,6))
    grouped = filtered_df.groupby(group_field)[numeric_field].mean()
    grouped.plot(kind='bar')
    plt.title(f'Mean of {numeric_field} grouped by {group_field}')
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.show()

    # Distribution of the numeric field
    plt.figure(figsize=(8,6))
    filtered_df[numeric_field].hist(bins=10)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()
else:
    print("No numeric or grouping field to visualize.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated how to load FAIR^2 dataset metadata and records using `mlcroissant`, explore available record sets, and perform basic EDA and visualization by referencing all dataset elements by their `@id`.

- All record sets, fields, columns were accessed dynamically via their Croissant `@id`
- Data was extracted into pandas DataFrames for processing
- Numeric and text fields used for normalization, filtering, and grouping
- Example visualizations showed the distributions and group means

*To extend this analysis: further explore other record sets or fields; apply clinical or molecular data filters; and implement more domain-specific analyses using the dataset metadata!*